# Dataset Audit — VoiceBank+DEMAND (Speech Enhancement)

This notebook **only inspects** your dataset. It does not move, delete, or modify any files.
Run every cell top to bottom. At the end you'll get a summary report + a CSV listing any
problem files, which we'll use to decide the cleaning strategy in the next step.

**Before running:** update the `DATA_ROOT` path below to point at your `data/raw` folder.


In [1]:
import os
os.system('pip install soundfile numpy pandas tqdm -q')


0

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm import tqdm


DATA_ROOT = Path(r"data/raw")

def resolve_wav_dir(base_path):
    """Return the folder that actually contains the .wav files,
    handling the common VoiceBank+DEMAND double-nested extraction."""
    if list(base_path.glob("*.wav")):
        return base_path
    nested = base_path / base_path.name
    if nested.exists() and list(nested.glob("*.wav")):
        return nested
    # fallback: search one level deep for any folder with wavs
    for sub in base_path.iterdir():
        if sub.is_dir() and list(sub.glob("*.wav")):
            return sub
    return base_path  # give up, return as-is (will show 0 files, flagging the problem)

FOLDERS = {
    "clean_train": resolve_wav_dir(DATA_ROOT / "clean_trainset_28spk_wav"),
    "noisy_train": resolve_wav_dir(DATA_ROOT / "noisy_trainset_28spk_wav"),
    "clean_test":  resolve_wav_dir(DATA_ROOT / "clean_testset_wav"),
    "noisy_test":  resolve_wav_dir(DATA_ROOT / "noisy_testset_wav"),
}

for name, path in FOLDERS.items():
    print(f"{name:12s} -> {path}  ({len(list(path.glob('*.wav')))} wav files)")

clean_train  -> data\raw\clean_trainset_28spk_wav  (11572 wav files)
noisy_train  -> data\raw\noisy_trainset_28spk_wav  (11572 wav files)
clean_test   -> data\raw\clean_testset_wav  (824 wav files)
noisy_test   -> data\raw\noisy_testset_wav  (824 wav files)


## 1. File counts & filename matching

Checks that every noisy file has a matching clean file with the *same filename*, and vice versa.

In [3]:
def list_wavs(folder):
    return sorted([f.name for f in folder.glob("*.wav")])

file_lists = {name: list_wavs(path) for name, path in FOLDERS.items()}

for name, files in file_lists.items():
    print(f"{name:12s}: {len(files)} files")


clean_train : 11572 files
noisy_train : 11572 files
clean_test  : 824 files
noisy_test  : 824 files


In [4]:
def compare_pairs(clean_key, noisy_key, label):
    clean_set = set(file_lists[clean_key])
    noisy_set = set(file_lists[noisy_key])

    only_clean = clean_set - noisy_set
    only_noisy = noisy_set - clean_set

    print(f"--- {label} ---")
    print(f"  clean count: {len(clean_set)}, noisy count: {len(noisy_set)}")
    print(f"  files only in clean (missing noisy pair): {len(only_clean)}")
    print(f"  files only in noisy (missing clean pair): {len(only_noisy)}")
    if only_clean:
        print("   e.g.", list(only_clean)[:5])
    if only_noisy:
        print("   e.g.", list(only_noisy)[:5])
    print()
    return only_clean, only_noisy

train_only_clean, train_only_noisy = compare_pairs("clean_train", "noisy_train", "TRAIN split")
test_only_clean, test_only_noisy = compare_pairs("clean_test", "noisy_test", "TEST split")


--- TRAIN split ---
  clean count: 11572, noisy count: 11572
  files only in clean (missing noisy pair): 0
  files only in noisy (missing clean pair): 0

--- TEST split ---
  clean count: 824, noisy count: 824
  files only in clean (missing noisy pair): 0
  files only in noisy (missing clean pair): 0



## 2. Per-file audit (sample rate, duration, channels, corruption, silence, clipping)

This reads every file's header + waveform once. On a few thousand files this can take a
couple of minutes — that's normal, `soundfile` reads efficiently.


In [5]:
def audit_folder(folder, label, silence_thresh_db=-50.0, clip_thresh=0.999):
    # Imported locally (not just at the top of the notebook) so this function
    # can't silently mark every file "corrupted" with error="name 'sf' is not
    # defined" if it's ever run in a fresh/restarted kernel without re-running
    # the earlier import cell first — that's exactly what produced the
    # garbage dataset_audit_report.csv from the previous run.
    import numpy as np
    import soundfile as sf

    rows = []
    files = list(folder.glob("*.wav"))
    for f in tqdm(files, desc=label):
        row = {"split": label, "filename": f.name, "path": str(f)}
        try:
            info = sf.info(str(f))
            row["samplerate"] = info.samplerate
            row["channels"] = info.channels
            row["duration_sec"] = info.duration
            row["corrupted"] = False

            # only read the waveform if the header looks sane, to compute loudness/clipping
            data, sr = sf.read(str(f))
            if data.ndim > 1:
                data = data.mean(axis=1)  # collapse to mono for stats

            if len(data) == 0:
                row["corrupted"] = True
                row["rms_db"] = None
                row["peak_abs"] = None
                row["is_silent"] = True
                row["is_clipped"] = False
            else:
                rms = np.sqrt(np.mean(data.astype(np.float64) ** 2))
                rms_db = 20 * np.log10(rms + 1e-12)
                peak = np.max(np.abs(data))
                row["rms_db"] = rms_db
                row["peak_abs"] = float(peak)
                row["is_silent"] = bool(rms_db < silence_thresh_db)
                row["is_clipped"] = bool(peak >= clip_thresh)

        except Exception as e:
            row["corrupted"] = True
            row["samplerate"] = None
            row["channels"] = None
            row["duration_sec"] = None
            row["rms_db"] = None
            row["peak_abs"] = None
            row["is_silent"] = None
            row["is_clipped"] = None
            row["error"] = str(e)

        rows.append(row)
    return pd.DataFrame(rows)

dfs = []
for name, path in FOLDERS.items():
    if path.exists():
        dfs.append(audit_folder(path, name))

audit_df = pd.concat(dfs, ignore_index=True)
audit_df.head()


,split,filename,path,samplerate,channels,duration_sec,corrupted,rms_db,peak_abs,is_silent,is_clipped
0,clean_train,p226_001.wav,data\raw\clean_trainset_28spk_wav\p226_001.wav,48000,1,2.280000,False,-23.744217,0.5,False,False
1,clean_train,p226_002.wav,data\raw\clean_trainset_28spk_wav\p226_002.wav,48000,1,3.900000,False,-26.707850,0.5,False,False
2,clean_train,p226_003.wav,data\raw\clean_trainset_28spk_wav\p226_003.wav,48000,1,7.774312,False,-26.342809,0.5,False,False
3,clean_train,p226_004.wav,data\raw\clean_trainset_28spk_wav\p226_004.wav,48000,1,5.190000,False,-25.027566,0.5,False,False
4,clean_train,p226_005.wav,data\raw\clean_trainset_28spk_wav\p226_005.wav,48000,1,7.420958,False,-25.185753,0.5,False,False


## 3. Summary report

In [6]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)

for split in audit_df["split"].unique():
    sub = audit_df[audit_df["split"] == split]
    print(f"\n[{split}]  total files: {len(sub)}")
    print(f"  corrupted/unreadable : {sub['corrupted'].sum()}")
    print(f"  sample rates found   : {sorted(sub['samplerate'].dropna().unique())}")
    print(f"  channel counts found : {sorted(sub['channels'].dropna().unique())}")
    print(f"  duration min/mean/max (sec): "
          f"{sub['duration_sec'].min():.2f} / {sub['duration_sec'].mean():.2f} / {sub['duration_sec'].max():.2f}")
    print(f"  silent clips (< -50dB RMS) : {sub['is_silent'].sum()}")
    print(f"  clipped clips (peak>=0.999): {sub['is_clipped'].sum()}")

print("\n" + "=" * 60)
print("FILENAME MISMATCHES")
print("=" * 60)
print(f"train: only-in-clean={len(train_only_clean)}, only-in-noisy={len(train_only_noisy)}")
print(f"test : only-in-clean={len(test_only_clean)}, only-in-noisy={len(test_only_noisy)}")


SUMMARY



[clean_train]  total files: 11572
  corrupted/unreadable : 0
  sample rates found   : [np.int64(48000)]
  channel counts found : [np.int64(1)]
  duration min/mean/max (sec): 1.09 / 2.92 / 15.11
  silent clips (< -50dB RMS) : 0
  clipped clips (peak>=0.999): 0

[noisy_train]  total files: 11572
  corrupted/unreadable : 0
  sample rates found   : [np.int64(48000)]
  channel counts found : [np.int64(1)]
  duration min/mean/max (sec): 1.09 / 2.92 / 15.11
  silent clips (< -50dB RMS) : 0
  clipped clips (peak>=0.999): 2

[clean_test]  total files: 824
  corrupted/unreadable : 0
  sample rates found   : [np.int64(48000)]
  channel counts found : [np.int64(1)]
  duration min/mean/max (sec): 1.24 / 2.51 / 9.77
  silent clips (< -50dB RMS) : 0
  clipped clips (peak>=0.999): 0

[noisy_test]  total files: 824
  corrupted/unreadable : 0
  sample rates found   : [np.int64(48000)]
  channel counts found : [np.int64(1)]
  duration min/mean/max (sec): 1.24 / 2.51 / 9.77
  silent clips (< -50dB RMS) :

## 4. Duplicate detection (identical audio content, not just filename)

In [7]:
import hashlib

def file_hash(path, block_size=65536):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block_size), b""):
            h.update(chunk)
    return h.hexdigest()

print("Hashing files to find exact duplicates (this reads raw bytes, fast)...")
audit_df["hash"] = [file_hash(p) for p in tqdm(audit_df["path"])]

dupe_groups = audit_df[audit_df.duplicated("hash", keep=False)].sort_values("hash")
print(f"\nExact duplicate files found: {dupe_groups['hash'].nunique()} groups, "
      f"{len(dupe_groups)} files total")
if len(dupe_groups):
    display(dupe_groups[["split", "filename", "hash"]].head(20))


Hashing files to find exact duplicates (this reads raw bytes, fast)...



Exact duplicate files found: 0 groups, 0 files total


## 5. Save the audit report

This CSV is the input to the cleaning step — nothing has been changed on disk yet.

In [8]:
audit_df.to_csv("dataset_audit_report.csv", index=False)
print("Saved: dataset_audit_report.csv")

problems = audit_df[
    audit_df["corrupted"].fillna(False)
    | audit_df["is_silent"].fillna(False)
    | audit_df["is_clipped"].fillna(False)
]
problems.to_csv("dataset_audit_problems_only.csv", index=False)
print(f"Saved: dataset_audit_problems_only.csv ({len(problems)} flagged rows)")


Saved: dataset_audit_report.csv
Saved: dataset_audit_problems_only.csv (2 flagged rows)
